# Datathon 2026 — BERTurk fine-tune (5-fold OOF meta-feature)

`dbmdz/bert-base-turkish-cased` hedefe fine-tune → fold-OOF + test tahmini. Donuk e5'ten farkı:
göreve adapte (text→score öğrenir), tek-çıktı (Ridge-768 şişmesi yok) → köprü daha iyi tutmalı.

## ⚠️ GPU KURULUMU — SIRAYLA YAP (bu sefer doğru):
1. **Settings → Accelerator → GPU T4 ×2**
2. **Settings → Internet → ON**  (BERTurk ağırlıkları HF'den iner)
3. **Session RESTART** — accelerator değişince ŞART (sağ üst güç/⟳ ikonu → Restart, veya Run → Restart & Clear)
4. Bu hücreyi çalıştır → **ilk çıktı `GPU: Tesla T4` olmalı.** `AssertionError` alırsan GPU bağlı değil, 3'e dön.

## Çıktı (Run sonrası /kaggle/working'den indir, bana yükle):
- `oof_berturk_train.npy` (train OOF) + `berturk_test.npy` (test tahmini)
Ensemble entegrasyonunu (sabit-ağırlık blend) lokalde yapacağız — OOF'a göre ağırlık TUNE edilmeyecek.

## Bana getireceğin (hücre çıktısının sonu):
- **BERTurk-only OOF** (düz | ağırlıklı) — donuk e5 embedding-only **düz 157.29 / ağırlıklı 180.35** ile kıyas
- Yıl-bazlı OOF (2025-2026 kritik)

In [ ]:
# ===================== BERTurk 5-fold fine-tune =====================
import os, glob, random, numpy as np, pandas as pd, torch
assert torch.cuda.is_available(), \
    "GPU YOK! Settings->Accelerator->GPU T4 -> Internet ON -> session RESTART -> tekrar çalıştır."
print("GPU:", torch.cuda.get_device_name(0))

SEED, N_SPLITS = 42, 5
MAX_LEN, LR, MIN_EPOCHS, MAX_EPOCHS, BATCH, PATIENCE = 256, 2e-5, 2, 3, 16, 1
# MIN_EPOCHS=2: her fold EN AZ 2 tam epoch eğitir (ep0,ep1 break edilmez).
# MAX_EPOCHS=3: best-of-3. Early stop yalnız ep>=MIN_EPOCHS sonrası + PATIENCE ardışık iyileşmeme.
# Not: ilk koşuda 'SEÇİLEN ep' çoğu fold'da ep2 ise val hâlâ düşüyordur -> MAX_EPOCHS=4 dene.
MODEL = 'dbmdz/bert-base-turkish-cased'
ID, TARGET, TEXT, YEAR = 'student_id', 'career_success_score', 'mentor_feedback_text', 'application_year'
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False  # reproducibility

# ---------- Veri + fold'lar (Faz 1-4 ile BİREBİR) ----------
base = [p for p in glob.glob('/kaggle/input/*') if os.path.exists(f'{p}/train.csv')]
if not base:
    hits = glob.glob('/kaggle/input/**/train.csv', recursive=True)
    assert hits, "train.csv yok — Add Input → Competitions → Datathon 2026"
    base = [os.path.dirname(hits[0])]
base = base[0]
train = pd.read_csv(f'{base}/train.csv'); test = pd.read_csv(f'{base}/test_x.csv')
y = train[TARGET].values.astype('float32')
texts_tr = train[TEXT].fillna('').tolist(); texts_te = test[TEXT].fillna('').tolist()

from sklearn.model_selection import StratifiedKFold
# strat'ı float64 hedeften hesapla -> Faz 1-5 fold'larıyla BİREBİR garanti (float32 cast'ten bağımsız)
tbin = pd.qcut(train[TARGET].values, 10, labels=False, duplicates='drop')
strat = train[YEAR].astype(str) + '_' + pd.Series(tbin).astype(str)
folds = list(StratifiedKFold(N_SPLITS, shuffle=True, random_state=SEED).split(train, strat))

tr_prop = train[YEAR].value_counts(normalize=True); te_prop = test[YEAR].value_counts(normalize=True)
w = np.nan_to_num(train[YEAR].map(lambda yr: te_prop.get(yr,0.0)/tr_prop.get(yr,np.nan)).values)

# ---------- Tokenize (bir kez) ----------
from transformers import AutoTokenizer, AutoModelForSequenceClassification
tok = AutoTokenizer.from_pretrained(MODEL)
def enc_all(texts):
    return tok(texts, truncation=True, max_length=MAX_LEN, padding='max_length', return_tensors='pt')
E_tr = enc_all(texts_tr); E_te = enc_all(texts_te)
ids_tr, am_tr = E_tr['input_ids'], E_tr['attention_mask']
ids_te, am_te = E_te['input_ids'], E_te['attention_mask']
dev = 'cuda'

def predict(model, ids, am, bs=64):
    model.eval(); out = []
    with torch.no_grad(), torch.cuda.amp.autocast():
        for s in range(0, len(ids), bs):
            o = model(input_ids=ids[s:s+bs].to(dev), attention_mask=am[s:s+bs].to(dev)).logits.squeeze(-1)
            out.append(o.float().cpu().numpy())
    return np.concatenate(out)

oof = np.zeros(len(train)); test_pred = np.zeros(len(test))
for fi, (tr, va) in enumerate(folds):
    mu, sd = float(y[tr].mean()), float(y[tr].std())          # hedef standardize (stabil eğitim)
    ytr = torch.tensor((y[tr]-mu)/sd, dtype=torch.float32)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=1).to(dev)
    opt = torch.optim.AdamW(model.parameters(), lr=LR)
    scaler = torch.cuda.amp.GradScaler()
    best_mse, best_pred, best_state, best_ep, no_improve = 1e9, None, None, -1, 0
    for ep in range(MAX_EPOCHS):
        model.train(); perm = torch.randperm(len(tr))
        for s in range(0, len(tr), BATCH):
            idx = perm[s:s+BATCH]
            gi = ids_tr[tr][idx].to(dev); ga = am_tr[tr][idx].to(dev); yb = ytr[idx].to(dev)
            opt.zero_grad()
            with torch.cuda.amp.autocast():
                pr = model(input_ids=gi, attention_mask=ga).logits.squeeze(-1)
                loss = ((pr - yb)**2).mean()
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        vp = predict(model, ids_tr[va], am_tr[va]) * sd + mu
        vmse = float(np.mean((y[va]-vp)**2))
        if vmse < best_mse - 1e-4:                            # iyileşme -> best güncelle, sayaç sıfırla
            best_mse, best_pred, best_ep = vmse, vp.copy(), ep
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
        print(f"fold{fi} ep{ep} valMSE={vmse:.3f}  (best ep{best_ep}={best_mse:.3f}, no_improve={no_improve})")
        # EARLY STOP: yalnız >=MIN_EPOCHS tam epoch sonrası (ep0,ep1 garanti çalışır) + PATIENCE ardışık iyileşmeme
        if ep >= MIN_EPOCHS and no_improve >= PATIENCE:
            print(f"  -> early stop (ep{ep}, {PATIENCE} epoch iyileşme yok)")
            break
    oof[va] = np.clip(best_pred, 0, 100)                      # OOF: SEÇİLEN epoch'un val tahmini
    model.load_state_dict(best_state)
    test_pred += np.clip(predict(model, ids_te, am_te) * sd + mu, 0, 100) / N_SPLITS
    del model; torch.cuda.empty_cache()
    print(f"  fold{fi} bitti: SEÇİLEN ep{best_ep} valMSE={best_mse:.3f}")

# ---------- BERTurk-only rapor ----------
def wmse(p): return float(np.sum(w*(y-p)**2)/np.sum(w))
plain = float(np.mean((y-oof)**2)); weighted = wmse(oof)
by_year = pd.DataFrame({'year': train[YEAR], 'e2': (y-oof)**2}).groupby('year')['e2'].mean()
print("\n================ BERTurk-only OOF ================")
print(f"düz={plain:.4f} | ağırlıklı={weighted:.4f}")
print(f"(donuk e5 embedding-only: düz 157.29 / ağırlıklı 180.35 — fine-tune bunu BELİRGİN geçmeli)")
print(by_year.round(3).to_string())

np.save('/kaggle/working/oof_berturk_train.npy', oof.astype('float32'))
np.save('/kaggle/working/berturk_test.npy', test_pred.astype('float32'))
print("\n[kaydedildi] oof_berturk_train.npy + berturk_test.npy -> indir, bana yükle")
